# Part 4 – Change Detection Core

**Branch:** `feature/change-detection-core`


## 10. Change-Detection Configuration & Functions

Order of use: camera-shift alignment → pixel-change map → box matching (Add/Delete/Move candidates) →
evidence scoring (pixel change, ghost check, move split/merge) → final decision (with low-confidence fallback).
All rules are hand-written; no classifier is trained anywhere in this stage.

In [ ]:

CONF_THRESH         = 0.4     # a box needs this YOLO confidence to CREATE a change event
LOW_CONF_THRESH     = 0.10    # boxes between LOW_CONF_THRESH and CONF_THRESH are kept only as evidence
                              # (ghost check below, and the low-confidence fallback pass)
MOVE_MIN_THRESH     = 15.0    # minimum displacement (px) that counts as a Move
MOVE_SAFETY_FACTOR  = 3.0     # move threshold = max(MOVE_MIN_THRESH, MOVE_SAFETY_FACTOR * camera-shift residual)
MOVE_REL_FRAC       = 0.20    # ...and the centre must also move more than this fraction of the box diagonal,
                              # so a 20px box-jitter on a 900px table is not reported as a Move
MAX_MATCH_DIST_FRAC = 0.35    # max centre distance for a match, as a fraction of the image diagonal
MIN_APPEARANCE_SIM  = 0.2     # min HSV-histogram correlation for two boxes to be the same object
W_DIST, W_IOU, W_APP = 0.55, 0.20, 0.25   # Hungarian cost weights (v40 used these but never defined them)

RECOVERY_APPEARANCE_SIM = 0.55   # 2nd-chance same-class match: if two leftover same-class boxes
                                  # (rejected by the strict distance cap above) look like the same
                                  # object, treat them as ONE Move instead of a Delete + Add pair.
CROSS_CLASS_SUPPRESS_IOU = 0.50  # if a leftover reference box and a leftover query box of a
                                  # DIFFERENT class overlap this much (post-alignment), it's almost
                                  # certainly the detector flipping its label on the same physical
                                  # object -- not a real Add+Delete -- so both are suppressed.
CHANGE_PROB_THRESHOLD = 0.60    # an event only counts as a genuine, final change if its
                                  # change-probability is >= this. Everything below is treated as
                                  # detector noise/jitter, not reported as a change.

CHANGE_MAP_WIDTH = 480     # the aligned reference/query pair is compared at this width
SSIM_FLOOR       = 0.10    # a pixel is "changed" only if its (1 - SSIM)/2 exceeds this ...
CHROMA_FLOOR     = 10.0    # ... or its a*b* colour difference (CIELAB units) exceeds this ...
PIXEL_CHANGE_K   = 3.0     # ... AND it is an outlier: robust z-score (median/MAD over the image) > K
VISUAL_SAT       = 0.35    # a box whose changed-pixel fraction reaches this gets the full visual score
MIN_BOX_CHANGE   = 0.10    # below this changed-pixel fraction a location counts as visually unchanged
GHOST_IOU        = 0.50    # a weak same-class box at the same spot in the other image means the object
                           # is still there (detector miss), so the Add/Delete is penalised ...
GHOST_PENALTY    = 0.80    # ... by this factor

DEFAULT_PARAMS = dict(
    conf_thresh=CONF_THRESH, low_conf_thresh=LOW_CONF_THRESH,
    move_min_thresh=MOVE_MIN_THRESH, move_safety_factor=MOVE_SAFETY_FACTOR, move_rel_frac=MOVE_REL_FRAC,
    max_match_dist_frac=MAX_MATCH_DIST_FRAC, min_appearance_sim=MIN_APPEARANCE_SIM,
    w_dist=W_DIST, w_iou=W_IOU, w_app=W_APP,
    recovery_sim=RECOVERY_APPEARANCE_SIM, cross_class_suppress_iou=CROSS_CLASS_SUPPRESS_IOU,
    prob_threshold=CHANGE_PROB_THRESHOLD,
    pixel_k=PIXEL_CHANGE_K, visual_sat=VISUAL_SAT, min_box_change=MIN_BOX_CHANGE,
    ghost_iou=GHOST_IOU, ghost_penalty=GHOST_PENALTY,
    alignment='ratio',     # 'ratio' = new ratio-test + sanity-checked homography, 'v40' = old alignment
    use_visual=True,       # score every event with pixel-change evidence
    use_ghost=True,        # veto Add/Delete when a weak detection shows the object is still there
    use_move_rules=True,   # split one-sided Moves into Add/Delete, merge same-class Add+Delete into a Move
    use_fallback=True,     # if nothing passes, re-run with the weak boxes (pixel evidence must carry it)
)


BASELINE_PARAMS = {**DEFAULT_PARAMS, 'alignment': 'v40', 'move_rel_frac': 0.0, 'use_visual': False, 'use_ghost': False,
                   'use_move_rules': False, 'use_fallback': False}

print(f"CONF_THRESH={CONF_THRESH}  LOW_CONF_THRESH={LOW_CONF_THRESH}  MOVE_MIN_THRESH={MOVE_MIN_THRESH}  "
      f"MAX_MATCH_DIST_FRAC={MAX_MATCH_DIST_FRAC}  RECOVERY_APPEARANCE_SIM={RECOVERY_APPEARANCE_SIM}  "
      f"CROSS_CLASS_SUPPRESS_IOU={CROSS_CLASS_SUPPRESS_IOU}  CHANGE_PROB_THRESHOLD={CHANGE_PROB_THRESHOLD}")
print(f"PIXEL_CHANGE_K={PIXEL_CHANGE_K}  VISUAL_SAT={VISUAL_SAT}  MIN_BOX_CHANGE={MIN_BOX_CHANGE}  "
      f"GHOST_IOU={GHOST_IOU}  MOVE_REL_FRAC={MOVE_REL_FRAC}")

In [ ]:
def _as_gray(img_or_path):
    if isinstance(img_or_path, str):
        return cv2.imread(img_or_path, cv2.IMREAD_GRAYSCALE)
    if img_or_path is None:
        return None
    return cv2.cvtColor(img_or_path, cv2.COLOR_BGR2GRAY) if img_or_path.ndim == 3 else img_or_path

def _homography_is_plausible(H, ref_shape, qry_shape, min_area_ratio=0.5, max_area_ratio=2.0):
    h, w = ref_shape[:2]
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    warped = cv2.perspectiveTransform(corners, H).reshape(-1, 2).astype(np.float32)
    if not np.all(np.isfinite(warped)) or not cv2.isContourConvex(warped):
        return False
    area_ratio = cv2.contourArea(warped) / float(qry_shape[0] * qry_shape[1])
    return min_area_ratio <= area_ratio <= max_area_ratio

def estimate_camera_shift_v40(reference, query, max_features=2000, good_match_pct=0.2,
                              ransac_reproj_thresh=5.0):
    img1, img2 = _as_gray(reference), _as_gray(query)
    if img1 is None or img2 is None:
        return np.eye(3), 0.0
    orb = cv2.ORB_create(max_features)
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)
    if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
        return np.eye(3), 0.0
    matches = sorted(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False).match(des1, des2),
                     key=lambda m: m.distance)
    matches = matches[:max(15, int(len(matches) * good_match_pct))]
    if len(matches) < 8:
        return np.eye(3), 0.0
    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransacReprojThreshold=ransac_reproj_thresh)
    if H is None:
        return np.eye(3), 0.0
    inliers = inlier_mask.ravel().astype(bool)
    if inliers.sum() < 8:
        return np.eye(3), 0.0
    pts1_proj = cv2.perspectiveTransform(pts1[inliers], H)
    return H, float(np.median(np.linalg.norm((pts1_proj - pts2[inliers]).reshape(-1, 2), axis=1)))

def estimate_camera_shift(reference, query, max_features=4000, ratio=0.75, ransac_reproj_thresh=5.0):
    img1, img2 = _as_gray(reference), _as_gray(query)
    if img1 is None or img2 is None:
        return np.eye(3), 0.0

    orb = cv2.ORB_create(max_features)
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)
    if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
        return np.eye(3), 0.0

    knn = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(des1, des2, k=2)
    matches = [p[0] for p in knn if len(p) == 2 and p[0].distance < ratio * p[1].distance]
    if len(matches) < 12:
        return np.eye(3), 0.0

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    H, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransacReprojThreshold=ransac_reproj_thresh)
    if H is None:
        return np.eye(3), 0.0

    inliers = inlier_mask.ravel().astype(bool)
    if inliers.sum() < 8 or not _homography_is_plausible(H, img1.shape, img2.shape):
        return np.eye(3), 0.0

    pts1_proj = cv2.perspectiveTransform(pts1[inliers], H)
    residuals = np.linalg.norm((pts1_proj - pts2[inliers]).reshape(-1, 2), axis=1)
    residual_px = float(np.median(residuals))

    return H, residual_px

def adaptive_move_thresh(residual_px, min_thresh=15.0, safety_factor=3.0):
    return max(min_thresh, safety_factor * residual_px)

### 10b. Pixel-level change map

The reference image is warped into the query frame with `H`, both are downscaled to 480 px wide, and each pixel gets
a dissimilarity from **SSIM** (structure, insensitive to exposure changes) and **a\*b\* chroma distance** (colour,
insensitive to lightness/shadows). Each dissimilarity is turned into a robust z-score using the median and MAD of the
whole image: most of an indoor scene is unchanged, so these statistics model noise + lighting drift, and real object
changes stand out as outliers. `box_change_fraction` then reports what fraction of a box's pixels changed.

In [ ]:
def _ssim_map(g1, g2, sigma=1.5):

    C1, C2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    g1, g2 = g1.astype(np.float32), g2.astype(np.float32)
    blur = lambda x: cv2.GaussianBlur(x, (0, 0), sigma)
    mu1, mu2 = blur(g1), blur(g2)
    s11 = blur(g1 * g1) - mu1 * mu1
    s22 = blur(g2 * g2) - mu2 * mu2
    s12 = blur(g1 * g2) - mu1 * mu2
    return ((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s11 + s22 + C2))

def _robust_z(x, valid):

    v = x[valid]
    if v.size == 0:
        return np.zeros_like(x)
    med = float(np.median(v))
    mad = float(np.median(np.abs(v - med))) * 1.4826
    return (x - med) / max(mad, 1e-3)

def compute_change_map(reference_bgr, query_bgr, H, work_width=CHANGE_MAP_WIDTH,
                        ssim_floor=SSIM_FLOOR, chroma_floor=CHROMA_FLOOR):

    if reference_bgr is None or query_bgr is None:
        return None
    hq, wq = query_bgr.shape[:2]
    scale = work_width / float(wq)
    size = (work_width, max(1, int(round(hq * scale))))
    s_ref = work_width / float(reference_bgr.shape[1])

    qry_small = cv2.resize(query_bgr, size, interpolation=cv2.INTER_AREA)
    ref_small = cv2.resize(reference_bgr, None, fx=s_ref, fy=s_ref, interpolation=cv2.INTER_AREA)
    H = np.eye(3) if H is None else H
    Hs = np.diag([scale, scale, 1.0]) @ H @ np.diag([1.0 / s_ref, 1.0 / s_ref, 1.0])
    ref_warp = cv2.warpPerspective(ref_small, Hs, size, flags=cv2.INTER_LINEAR)
    valid = cv2.warpPerspective(np.full(ref_small.shape[:2], 255, np.uint8), Hs, size,
                                flags=cv2.INTER_NEAREST)
    valid = cv2.erode(valid, np.ones((7, 7), np.uint8)) > 0

    ref_warp = cv2.GaussianBlur(ref_warp, (0, 0), 1.0)
    qry_small = cv2.GaussianBlur(qry_small, (0, 0), 1.0)


    d_struct = (1.0 - _ssim_map(cv2.cvtColor(ref_warp, cv2.COLOR_BGR2GRAY),
                                cv2.cvtColor(qry_small, cv2.COLOR_BGR2GRAY))) / 2.0

    lab1 = cv2.cvtColor(ref_warp, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab2 = cv2.cvtColor(qry_small, cv2.COLOR_BGR2LAB).astype(np.float32)
    d_chroma = np.sqrt((lab1[..., 1] - lab2[..., 1]) ** 2 + (lab1[..., 2] - lab2[..., 2]) ** 2)
    d_chroma = cv2.GaussianBlur(d_chroma, (0, 0), 2.0)

    z_struct = np.where(d_struct > ssim_floor, _robust_z(d_struct, valid), 0.0)
    z_chroma = np.where(d_chroma > chroma_floor, _robust_z(d_chroma, valid), 0.0)
    score = np.maximum(z_struct, z_chroma)
    score[~valid] = 0.0
    return {'score': np.clip(score, 0, 60000).astype(np.float16), 'scale': scale}

def box_change_fraction(change, bbox_query_frame, pixel_k):

    if change is None:
        return None
    sc, s = change['score'], change['scale']
    h, w = sc.shape
    x1, y1, x2, y2 = bbox_query_frame
    x1, y1 = max(0, int(np.floor(x1 * s))), max(0, int(np.floor(y1 * s)))
    x2, y2 = min(w, int(np.ceil(x2 * s))), min(h, int(np.ceil(y2 * s)))
    if x2 <= x1 or y2 <= y1:
        return 0.0
    return float((sc[y1:y2, x1:x2] > pixel_k).mean())

def changed_mask(change, pixel_k):
    return None if change is None else (change['score'] > pixel_k)

In [ ]:
def compute_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x3, y3, x4, y4 = box2
    xi1, yi1 = max(x1, x3), max(y1, y3)
    xi2, yi2 = min(x2, x4), min(y2, y4)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x4 - x3) * (y4 - y3)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def extract_boxes(results, model):
    boxes = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0].cpu().numpy())
            boxes.append({
                'class': model.names[cls_id],
                'bbox': xyxy,
                'confidence': float(box.conf[0].cpu().numpy())
            })
    return boxes

def normalize_name(s):
    return s.replace('-', ' ').replace('_', ' ').strip().lower()

def warp_bbox(bbox, H):
    if H is None:
        return bbox
    x1, y1, x2, y2 = bbox
    corners = np.array([[[x1, y1]], [[x2, y1]], [[x2, y2]], [[x1, y2]]], dtype=np.float32)
    warped = cv2.perspectiveTransform(corners, H).reshape(-1, 2)
    wx1, wy1 = float(warped[:, 0].min()), float(warped[:, 1].min())
    wx2, wy2 = float(warped[:, 0].max()), float(warped[:, 1].max())
    return [wx1, wy1, wx2, wy2]

def get_image_size(path):
    with PILImage.open(path) as im:
        return im.size

def crop_bbox(img, bbox, pad=4):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2]

def box_histogram(img_bgr, bbox, pad=4):
    crop = crop_bbox(img_bgr, bbox, pad)
    if crop is None or crop.size == 0:
        return None
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist

def attach_histograms(boxes, img_bgr):
    for b in boxes:
        b['hist'] = box_histogram(img_bgr, b['bbox'])
    return boxes

def appearance_similarity(box_a, box_b):
    if 'hist' not in box_a or 'hist' not in box_b:
        return 1.0
    if box_a['hist'] is None or box_b['hist'] is None:
        return 0.0
    return float(max(0.0, cv2.compareHist(box_a['hist'], box_b['hist'], cv2.HISTCMP_CORREL)))

def split_by_conf(boxes, conf_thresh, low_conf_thresh=0.0):
    strong = [b for b in boxes if b['confidence'] >= conf_thresh]
    weak = [b for b in boxes if low_conf_thresh <= b['confidence'] < conf_thresh]
    return strong, weak


def _clip01(x):
    return float(min(1.0, max(0.0, x)))

def _center(bbox):
    return (bbox[0] + bbox[2]) / 2.0, (bbox[1] + bbox[3]) / 2.0

def _diag(bbox):
    return float(np.hypot(bbox[2] - bbox[0], bbox[3] - bbox[1]))

def move_probability(dist, move_thresh, sim, conf_b, conf_a, visual=None, saturation_ratio=2.5):
    ratio = dist / max(move_thresh, 1e-6)
    dist_term = _clip01((ratio - 1.0) / max(saturation_ratio - 1.0, 1e-6))
    sim_term = _clip01(sim)
    conf_term = _clip01((conf_b + conf_a) / 2.0)
    if visual is None:
        p = 0.55 * dist_term + 0.30 * sim_term + 0.15 * conf_term
    else:
        p = 0.35 * dist_term + 0.15 * sim_term + 0.15 * conf_term + 0.35 * _clip01(visual)
    return _clip01(p)

def add_delete_probability(conf, cross_class_iou=0.0, visual=None, ghost=False, ghost_penalty=0.8):
    base = _clip01(conf)
    if visual is not None:
        base = 0.45 * base + 0.55 * _clip01(visual)
    p = base * (1.0 - 0.85 * _clip01(cross_class_iou))
    if ghost:
        p *= (1.0 - ghost_penalty)
    return _clip01(p)


def match_boxes(reference_boxes, query_boxes, H=None, move_thresh=50, image_diag=None, params=None,
                change=None, ref_weak=(), qry_weak=()):
    P = {**DEFAULT_PARAMS, **(params or {})}
    events, matched_pairs, suppressed = [], [], []
    max_match_dist = (P['max_match_dist_frac'] * image_diag) if image_diag else float('inf')
    use_visual = bool(P['use_visual']) and change is not None
    move_rules = use_visual and bool(P['use_move_rules'])
    use_appearance = all('hist' in b for b in list(reference_boxes) + list(query_boxes))

    ref_warped = [warp_bbox(b['bbox'], H) for b in reference_boxes]
    ref_weak_warped = [warp_bbox(b['bbox'], H) for b in ref_weak]

    # changed-pixel fraction at every box location (reference boxes are looked up at their warped position)
    ref_frac = [box_change_fraction(change, rw, P['pixel_k']) for rw in ref_warped] if use_visual \
        else [None] * len(reference_boxes)
    qry_frac = [box_change_fraction(change, a['bbox'], P['pixel_k']) for a in query_boxes] if use_visual \
        else [None] * len(query_boxes)

    def vis(frac):
        return None if frac is None else _clip01(frac / max(P['visual_sat'], 1e-6))

    def changed(frac):
        return frac is not None and frac >= P['min_box_change']

    def ghost_for_ref(i):
        cls = reference_boxes[i]['class']
        return any(w['class'] == cls and compute_iou(ref_warped[i], w['bbox']) >= P['ghost_iou']
                   for w in qry_weak)

    def ghost_for_qry(j):
        cls = query_boxes[j]['class']
        return any(w['class'] == cls and compute_iou(query_boxes[j]['bbox'], ww) >= P['ghost_iou']
                   for w, ww in zip(ref_weak, ref_weak_warped))

    def move_thresh_for(i, j):
        rel = P['move_rel_frac'] * 0.5 * (_diag(ref_warped[i]) + _diag(query_boxes[j]['bbox']))
        return max(move_thresh, rel)

    def delete_event(i, note=None):
        b = reference_boxes[i]
        ghost = bool(P['use_ghost']) and ghost_for_ref(i)
        cross_iou = max((compute_iou(ref_warped[i], query_boxes[j]['bbox'])
                         for j in unmatched_qry if query_boxes[j]['class'] != b['class']), default=0.0)
        e = {'type': 'Delete', 'object': b['class'], 'confidence': b['confidence'], 'movement': 0.0,
             'probability': add_delete_probability(b['confidence'], cross_iou, vis(ref_frac[i]),
                                                   ghost, P['ghost_penalty']),
             'visual_change': ref_frac[i], 'ghost': ghost, 'bbox_ref': b['bbox']}
        if note:
            e['note'] = note
        return e

    def add_event(j, note=None):
        a = query_boxes[j]
        ghost = bool(P['use_ghost']) and ghost_for_qry(j)
        cross_iou = max((compute_iou(ref_warped[i], a['bbox'])
                         for i in unmatched_ref if reference_boxes[i]['class'] != a['class']), default=0.0)
        e = {'type': 'Add', 'object': a['class'], 'confidence': a['confidence'], 'movement': 0.0,
             'probability': add_delete_probability(a['confidence'], cross_iou, vis(qry_frac[j]),
                                                   ghost, P['ghost_penalty']),
             'visual_change': qry_frac[j], 'ghost': ghost, 'bbox_qry': a['bbox']}
        if note:
            e['note'] = note
        return e

    def emit_match(i, j, dist, sim, note=None):
        b, a = reference_boxes[i], query_boxes[j]
        mt = move_thresh_for(i, j)
        role = 'unchanged'
        if dist > mt:
            role = 'Move'
            if move_rules:
                src, dst = changed(ref_frac[i]), changed(qry_frac[j])
                if src and not dst:
                    role = 'Delete'
                elif dst and not src:
                    role = 'Add'
                elif not src and not dst:
                    role = 'unchanged'
        if role == 'Move':
            v = 0.5 * (vis(ref_frac[i]) + vis(qry_frac[j])) if use_visual else None
            e = {'type': 'Move', 'object': b['class'],
                 'confidence': (b['confidence'] + a['confidence']) / 2, 'movement': dist,
                 'appearance_sim': sim,
                 'probability': move_probability(dist, mt, sim, b['confidence'], a['confidence'], v),
                 'visual_change': None if v is None else 0.5 * (ref_frac[i] + qry_frac[j]),
                 'bbox_ref': b['bbox'], 'bbox_qry': a['bbox']}
            if note:
                e['note'] = note
            events.append(e)
        elif role == 'Delete':
            events.append(delete_event(i, 'one-sided move: only the source location changed'))
        elif role == 'Add':
            events.append(add_event(j, 'one-sided move: only the destination location changed'))
        matched_pairs.append((b, a, dist, sim, role))

    classes = set(b['class'] for b in reference_boxes) | set(a['class'] for a in query_boxes)
    unmatched_ref = set(range(len(reference_boxes)))
    unmatched_qry = set(range(len(query_boxes)))


    for cls in classes:
        b_idx = [i for i, b in enumerate(reference_boxes) if b['class'] == cls]
        a_idx = [j for j, a in enumerate(query_boxes) if a['class'] == cls]
        if not b_idx or not a_idx:
            continue

        cost = np.zeros((len(b_idx), len(a_idx)))
        dist_cache = np.zeros_like(cost)
        iou_cache = np.zeros_like(cost)
        sim_cache = np.ones_like(cost)
        for bi, i in enumerate(b_idx):
            bx, by = _center(ref_warped[i])
            for aj, j in enumerate(a_idx):
                ax, ay = _center(query_boxes[j]['bbox'])
                dist = float(np.hypot(ax - bx, ay - by))
                dist_frac = min(dist / image_diag, 1.0) if image_diag else min(dist / 1000.0, 1.0)
                iou = compute_iou(ref_warped[i], query_boxes[j]['bbox'])
                sim = appearance_similarity(reference_boxes[i], query_boxes[j])
                dist_cache[bi, aj], iou_cache[bi, aj], sim_cache[bi, aj] = dist, iou, sim
                cost[bi, aj] = P['w_dist'] * dist_frac + P['w_iou'] * (1.0 - iou) + P['w_app'] * (1.0 - sim)

        row_ind, col_ind = linear_sum_assignment(cost)
        for bi, aj in zip(row_ind, col_ind):
            dist, sim, iou = float(dist_cache[bi, aj]), float(sim_cache[bi, aj]), float(iou_cache[bi, aj])
            if dist > max_match_dist:
                continue
            if sim < P['min_appearance_sim'] and iou < 0.15:
                continue
            i, j = b_idx[bi], a_idx[aj]
            unmatched_ref.discard(i)
            unmatched_qry.discard(j)
            emit_match(i, j, dist, sim)


    for cls in classes:
        b_left = sorted(i for i in unmatched_ref if reference_boxes[i]['class'] == cls)
        a_left = sorted(j for j in unmatched_qry if query_boxes[j]['class'] == cls)
        if not b_left or not a_left:
            continue
        scored = sorted(((appearance_similarity(reference_boxes[i], query_boxes[j]), i, j)
                         for i in b_left for j in a_left), key=lambda t: -t[0])
        for sim, i, j in scored:
            if i not in unmatched_ref or j not in unmatched_qry:
                continue
            recovered = use_appearance and sim >= P['recovery_sim']
            merged = (move_rules and changed(ref_frac[i]) and changed(qry_frac[j])
                      and not (P['use_ghost'] and (ghost_for_ref(i) or ghost_for_qry(j))))
            if not (recovered or merged):
                continue
            bx, by = _center(ref_warped[i])
            ax, ay = _center(query_boxes[j]['bbox'])
            dist = float(np.hypot(ax - bx, ay - by))
            unmatched_ref.discard(i)
            unmatched_qry.discard(j)
            emit_match(i, j, dist, sim,
                       note='recovered match (same object relocated beyond the strict search radius)'
                       if recovered else
                       'merged: same-class object vanished at one spot and appeared at another')


    cross_suppressed_ref, cross_suppressed_qry = set(), set()
    for i in sorted(unmatched_ref):
        best_iou, best_j = 0.0, None
        for j in unmatched_qry:
            if j in cross_suppressed_qry or query_boxes[j]['class'] == reference_boxes[i]['class']:
                continue
            iou = compute_iou(ref_warped[i], query_boxes[j]['bbox'])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j is not None and best_iou >= P['cross_class_suppress_iou']:
            cross_suppressed_ref.add(i)
            cross_suppressed_qry.add(best_j)
            suppressed.append({'ref_class': reference_boxes[i]['class'], 'qry_class': query_boxes[best_j]['class'],
                               'iou': round(best_iou, 3),
                               'note': 'cross-class overlap -- likely a detector class-flip on the same object, not a real change'})


    for i in sorted(unmatched_ref - cross_suppressed_ref):
        events.append(delete_event(i))
    for j in sorted(unmatched_qry - cross_suppressed_qry):
        events.append(add_event(j))

    return events, matched_pairs, suppressed


def decide_final_changes(events, prob_threshold=CHANGE_PROB_THRESHOLD):

    scored = [e for e in events if e['type'] in ('Add', 'Delete', 'Move')]
    changes = sorted((e for e in scored if e['probability'] >= prob_threshold),
                      key=lambda e: -e['probability'])
    rejected = sorted((e for e in scored if e['probability'] < prob_threshold),
                       key=lambda e: -e['probability'])

    if not changes:
        if rejected:
            top = rejected[0]
            summary = (f"No significant change detected -- closest candidate was '{top['object']}' "
                       f"({top['type']}) at {top['probability']*100:.0f}%, below the "
                       f"{prob_threshold*100:.0f}% decision threshold.")
        else:
            summary = "No significant change detected -- scene unchanged."
        return {'has_change': False, 'changes': [], 'rejected': rejected, 'summary': summary}

    parts = [f"'{e['object']}' -- {e['type']} ({e['probability']*100:.0f}%)" for e in changes]
    n = len(changes)
    summary = (f"{n} object change{'s' if n != 1 else ''} detected: " + "; ".join(parts))
    return {'has_change': True, 'changes': changes, 'rejected': rejected, 'summary': summary}


def detect_changes(c, params=None):
    P = {**DEFAULT_PARAMS, **(params or {})}
    H, residual = (c['H'], c['residual_px']) if P['alignment'] == 'ratio' else (c['H_v40'], c['residual_px_v40'])
    move_thresh = adaptive_move_thresh(residual, P['move_min_thresh'], P['move_safety_factor'])
    strong_r, weak_r = split_by_conf(c['reference_boxes'], P['conf_thresh'], P['low_conf_thresh'])
    strong_q, weak_q = split_by_conf(c['query_boxes'], P['conf_thresh'], P['low_conf_thresh'])
    change = c.get('change') if P['use_visual'] else None
    common = dict(H=H, move_thresh=move_thresh, image_diag=c['image_diag'], params=P, change=change)

    events, matched, suppressed = match_boxes(strong_r, strong_q, ref_weak=weak_r, qry_weak=weak_q, **common)
    decision = decide_final_changes(events, P['prob_threshold'])
    used_pass = 'strict'
    if not decision['has_change'] and P['use_fallback'] and change is not None and (weak_r or weak_q):
        ev2, m2, s2 = match_boxes(strong_r + weak_r, strong_q + weak_q, **common)
        d2 = decide_final_changes(ev2, P['prob_threshold'])
        if d2['has_change']:
            events, matched, suppressed, decision = ev2, m2, s2, d2
            used_pass = 'fallback (low-confidence boxes)'
    return {'events': events, 'matched_pairs': matched, 'suppressed': suppressed, 'decision': decision,
            'move_thresh': move_thresh, 'H': H, 'residual_px': residual, 'pass': used_pass, 'change': change,
            'reference_boxes': strong_r, 'query_boxes': strong_q}


def summarize_events(events, matched_pairs=None):
    matched_pairs = matched_pairs or []
    add_events = [e for e in events if e['type'] == 'Add']
    del_events = [e for e in events if e['type'] == 'Delete']
    move_events = [e for e in events if e['type'] == 'Move']

    def best(evs):
        return max(evs, key=lambda e: e['probability']) if evs else None

    best_add, best_del, best_move = best(add_events), best(del_events), best(move_events)
    max_matched_dist = max((m[2] for m in matched_pairs), default=0.0)
    avg_appearance_sim = float(np.mean([m[3] for m in matched_pairs])) if matched_pairs else 0.0

    return {
        'n_add': len(add_events), 'n_del': len(del_events), 'n_move': len(move_events),
        'net': len(add_events) - len(del_events),
        'best_add_conf': best_add['confidence'] if best_add else 0.0,
        'best_del_conf': best_del['confidence'] if best_del else 0.0,
        'best_move_conf': best_move['confidence'] if best_move else 0.0,
        'best_move_dist': best_move['movement'] if best_move else 0.0,
        'max_matched_dist': max_matched_dist,
        'avg_appearance_sim': avg_appearance_sim,
        'best_add': best_add, 'best_del': best_del, 'best_move': best_move,
    }


def primary_event(events):
    decision = decide_final_changes(events)
    return decision['changes'][0] if decision['has_change'] else None


def build_events_df(cache, params=None):
    P = {**DEFAULT_PARAMS, **(params or {})}
    event_rows, summary_rows, contexts = [], [], []
    for c in cache:
        r = detect_changes(c, P)
        events, decision = r['events'], r['decision']
        final_ids = {id(e) for e in decision['changes']}

        for e in events:
            event_rows.append({
                'pair_id': c['pair_id'], 'object': e['object'], 'change_type': e['type'],
                'confidence': round(e['confidence'], 3), 'movement_px': round(e['movement'], 1),
                'visual_change': None if e.get('visual_change') is None else round(e['visual_change'], 3),
                'probability': round(e['probability'], 3), 'is_final_change': id(e) in final_ids,
                'split': c['split'],
            })

        stats = summarize_events(events, r['matched_pairs'])
        pe = decision['changes'][0] if decision['has_change'] else None
        summary_rows.append({
            'pair_id': c['pair_id'],
            'true_change': c['true_change'], 'true_object': c['true_object'], 'room': c['room'],
            'split': c['split'],
            'predicted_change': pe['type'] if pe else 'None',
            'predicted_object': pe['object'] if pe else 'None',
            'predicted_change_prob': round(pe['probability'], 3) if pe else 0.0,
            'n_events': len(events), 'n_final_changes': len(decision['changes']),
            'n_suppressed_cross_class': len(r['suppressed']),
            'decision_pass': r['pass'],
            'move_thresh_used': r['move_thresh'],
            'camera_shift_residual_px': r['residual_px'],
        })
        contexts.append({'stats': stats, 'matched_pairs': r['matched_pairs'], 'events': events,
                         'suppressed': r['suppressed'], 'decision': decision, 'pass': r['pass']})
    return pd.DataFrame(event_rows), pd.DataFrame(summary_rows), contexts


CHANGE_LABELS = ['Add', 'Delete', 'Move']

def pair_metrics(summary_df):
    """Pair-level scores. A 'None' prediction counts as WRONG (every pair in this dataset has a change)."""
    y, p = summary_df['true_change'], summary_df['predicted_change']
    f1s = []
    for c in CHANGE_LABELS:
        tp = int(((y == c) & (p == c)).sum())
        fp = int(((y != c) & (p == c)).sum())
        fn = int(((y == c) & (p != c)).sum())
        f1s.append(2 * tp / max(2 * tp + fp + fn, 1))
    obj_ok = summary_df['predicted_object'].map(normalize_name) == summary_df['true_object'].map(normalize_name)
    return {'n_pairs': len(summary_df),
            'accuracy': float((y == p).mean()),
            'macro_f1': float(np.mean(f1s)),
            'coverage': float((p != 'None').mean()),
            'type_and_object_acc': float(((y == p) & obj_ok).mean())}